# PGA 3D — Plane-Based Geometric Algebra

**Part I · Geometric Algebra & Core** — Tutorial 07

This tutorial is a deep dive into `BasisPGA3`, the **plane-based** (projective)
geometric algebra of 3D Euclidean space. Where most algebras encode a *point* as a
grade-1 vector, PGA3 flips the encoding: a **plane is a vector**, a **line is a
bivector**, and a **point is a trivector**.

By the end you will be able to:

- Understand the **single-null-vector embedding** — `e0` with `e0² = 0` and its
  reciprocal `e0_recip`.
- Build **points, lines, and planes** as blades of grades 3, 2, and 1.
- Compute **intersections** with `meet()` — the meet of two planes is their
  intersection line, and the meet of three planes is their intersection point.
- Compute **unions** with `join()` — the join of two points is the line through
  them, and the join of a line and a point is their spanning plane.
- Contrast `meet()`/`join()` with the outer and inner products, and use the outer
  product as an **incidence** test.
- Construct **translators, rotors, and motors** and apply them with the sandwich
  product — unifying rotations, translations, and reflections in one framework.
- Use the `Geometry` class and the Gunn/Dorst `e0` / `e0_recip` pairing.

> **Prerequisites:** [Tutorial 03](../03_basis_classes/) (named blades) and
> [Tutorial 04](../04_euclidean_e3/) (rotors). Geometric entities and operators are
> created through the `pytanga.geometry` submodule.


## 1. Setup

The imports mirror the earlier deep dives, but with `BasisPGA3` and the
plane-based entity and operator types.


In [1]:
import math

from pytanga import MV
from pytanga.basis import BasisPGA3
from pytanga.geometry import (
    Direction,
    Geometry,
    Line,
    Motor,
    Plane,
    Point,
    ReflectionLine,
    ReflectionPlane,
    ReflectionPoint,
    Rotor,
    Space,
    Translator,
)

PGA = BasisPGA3()       # Gunn/Dorst 4D PGA via a 5D null-vector embedding
geo = Geometry(PGA)     # binds the algebra; OPNS/IPNS read from PGA.opns (default True)


## 2. The single-null-vector embedding

`BasisPGA3` implements the **Gunn/Dorst** plane-based PGA. It uses a single null
basis vector `e0` with `e0² = 0`. Because TANGA's Clifford algebra only supports
basis vectors that square to ±1, the null vector is modelled through a 5D embedding

    e0 → ep + em,    ep² = +1,  em² = −1

(see `PGA.ep` / `PGA.em`). The pair `(ep, em)` generates the 5D algebra
`G(5, 0b10000)`; the subspace `{e1, e2, e3, e0}` is algebraically isomorphic to the
Gunn/Dorst 4D PGA.

The null vector `e0` has a reciprocal `e0_recip = ½·ep − ½·em`, defined so that
`⟨e0 · e0_recip⟩₀ = 1`.


In [2]:
e1, e2, e3 = PGA.e1, PGA.e2, PGA.e3
e0 = PGA.e0
e0_recip = PGA.e0_recip

print("dim       =", PGA.dim)        # 5 — the 4D PGA subspace plus the embedding
print("signature =", bin(PGA.sig))   # 0b10000: em (= e5) squares to −1
print()

e0.show("e0   (the Gunn/Dorst null vector)")
print("e0 * e0   =", e0 * e0, "   (nilpotent: e0² = 0)")
print()
print("e0_recip  =", e0_recip, "   (= ½·ep − ½·em)")
print("⟨e0·e0_recip⟩₀ =", PGA.sp(e0, e0_recip))


dim       = 5
signature = 0b10000



e0   (the Gunn/Dorst null vector): e0

e0 * e0   = 0    (nilpotent: e0² = 0)

e0_recip  = e0i    (= ½·ep − ½·em)
⟨e0·e0_recip⟩₀ = 1.0


## 3. Points, lines, planes — the plane-based encoding

PGA3 is called *plane-based* because the **plane is the grade-1 element**:

| Entity | OPNS grade | Blade |
|---|---|---|
| Plane  | 1 | `n + d·e0` |
| Line   | 2 | wedge of two planes |
| Point  | 3 | wedge of three planes |
| Space  | 4 | `e1 ∧ e2 ∧ e3 ∧ e0` |

Compare this with the conformal model (Tutorial 06), where a point is grade 1 and
a sphere is grade 4: PGA3 shifts every entity and drops the extra null dimension.
A point is the intersection of three planes, a line the intersection of two.


In [3]:
# A plane is a grade-1 vector:  n + d·e0   (unit normal n, signed distance d)
z0 = e3            # the plane z = 0   (normal +e3, offset 0)
z3 = e3 - 3 * e0   # the plane z = 3   (normal +e3, offset 3)

z0.show("plane z = 0")
z3.show("plane z = 3")
print()
print("analyze(z0) ->", geo.analyze(z0))
print("analyze(z3) ->", geo.analyze(z3))


plane z = 0: e3

plane z = 3: - 3 e0 + e3


analyze(z0) -> Plane(pt=Point(-0.00, -0.00, -0.00), n=Dir(0.00, 0.00, 1.00))
analyze(z3) -> Plane(pt=Point(0.00, 0.00, 3.00), n=Dir(0.00, 0.00, 1.00))


In [4]:
# A line is the wedge of two planes (grade 2)
line = e1 ^ e2          # x = 0  ∧  y = 0  →  the z-axis
line.show("line = e1 ^ e2")
print("analyze(line) ->", geo.analyze(line), " grades", line.grades)
print()

# A point is the wedge of three planes (grade 3)
point = (e1 - 1 * e0) ^ (e2 - 2 * e0) ^ (e3 - 3 * e0)
point.show("point = (e1−e0) ∧ (e2−2e0) ∧ (e3−3e0)")
print("analyze(point) ->", geo.analyze(point), " grades", point.grades)
print()

# The origin is simply e1 ∧ e2 ∧ e3
origin = e1 ^ e2 ^ e3
print("analyze(origin) ->", geo.analyze(origin))


line = e1 ^ e2: e12

analyze(line) -> Line(org=Point(0.00, 0.00, 0.00), dir=Dir(0.00, -0.00, 1.00))  grades [2]



point = (e1−e0) ∧ (e2−2e0) ∧ (e3−3e0): e032 + 2 e013 + 3 e021 + e123

analyze(point) -> Point(1.00, 2.00, 3.00)  grades [3]

analyze(origin) -> Point(-0.00, -0.00, -0.00)


## 4. Entities from the geometry submodule

As everywhere in pytanga, geometric entities are created through the
`pytanga.geometry` submodule. The `Geometry` convenience class maps entity
dataclasses to multivectors and back with a single `geo(...)` call.


In [5]:
entities = [
    ("Point", Point(1, 2, 3)),
    ("Direction", Direction(0, 1, 0)),
    ("Line", Line(origin=Point(1, 0, 0), direction=Direction(0, 1, 0))),
    ("Plane", Plane(point=Point(0, 0, 3), normal=Direction(0, 0, 1))),
    ("Space", Space()),
]

for name, e in entities:
    mv = geo.create(e)
    result = geo.analyze(mv)
    print(f"{name:10s} grade {mv.grades} -> {result} ({type(result).__name__})")


Point      grade [3] -> Point(1.00, 2.00, 3.00) (Point)
Direction  grade [3] -> Dir(0.00, 1.00, 0.00) (Direction)
Line       grade [2] -> Line(org=Point(1.00, -0.00, 0.00), dir=Dir(0.00, 1.00, 0.00)) (Line)
Plane      grade [1] -> Plane(pt=Point(0.00, 0.00, 3.00), n=Dir(0.00, 0.00, 1.00)) (Plane)
Space      grade [4] -> Space(scale=1.0) (Space)


## 5. Incidence and the outer vs inner product

For OPNS blades in PGA3 the **outer product** `∧` is the incidence test: a point
lies on a plane exactly when their outer product vanishes. The **inner product** `|`
measures the metric (`e_i · e_j = δ_ij`, `e0 · e0 = 0`) — it does not give the
incidence relation here.


In [6]:
plx = e1                              # the plane x = 0
P_on  = geo.create(Point(0, 2, 3))   # x = 0 → on the plane
P_off = geo.create(Point(1, 2, 3))   # x = 1 → off the plane

print("P_on  ^ plx is zero:", (P_on  ^ plx).is_zero)   # True  — incident
print("P_off ^ plx is zero:", (P_off ^ plx).is_zero)   # False — not incident
print()

# The inner product encodes the metric
print("e1 | e1 =", e1 | e1)     # +1 (Euclidean metric)
print("e1 | e2 =", e1 | e2)     #  0 (orthogonal)
print("e0 | e0 =", e0 | e0)     #  0 (null vector)


P_on  ^ plx is zero: True
P_off ^ plx is zero: False

e1 | e1 = 1
e1 | e2 = 0
e0 | e0 = 0


## 6. meet() and join() — intersection and union

`meet()` and `join()` are the blade lattice operations of PGA3, following the
plane-based (Gunn/Dorst) convention directly:

- `meet(A, B)` is the **intersection** (the outer product) — the largest blade
  contained in both. The meet of two planes is their intersection line, and the
  meet of three planes is their intersection point.
- `join(A, B)` is the **union** (the regressive product `⋆(⋆A ∧ ⋆B)`) — the
  smallest blade containing both. The join of two points is the line through
  them, and the join of a line and a point is their spanning plane.

Because a plane is a grade-1 vector and a point is a grade-3 trivector, the two
operations are duals of each other: **meet** the planes to intersect them,
**join** the points to connect them.


In [7]:
plx = e1                               # plane x = 0
ply = e2                               # plane y = 0
plz = e3 - 2 * e0                     # plane z = 2

# meet() = intersection
L = PGA.meet(plx, ply)                 # x = 0 ∩ y = 0  →  the z-axis line
P = PGA.meet(PGA.meet(plx, ply), plz)  # ∩ z = 2        →  the point (0, 0, 2)
print("meet(plx, ply)           ->", geo.analyze(L), " grades", L.grades)
print("meet(meet(plx,ply), plz) ->", geo.analyze(P), " grades", P.grades)


meet(plx, ply)           -> Line(org=Point(0.00, 0.00, 0.00), dir=Dir(0.00, -0.00, 1.00))  grades [2]
meet(meet(plx,ply), plz) -> Point(-0.00, -0.00, 2.00)  grades [3]


In [8]:
pA = geo.create(Point(1, 0, 0))
pB = geo.create(Point(0, 1, 0))
pC = geo.create(Point(0, 0, 1))

# join() = union
line_ab   = PGA.join(pA, pB)           # the line through two points
plane_abc = PGA.join(line_ab, pC)      # the plane through three points
print("join(pA, pB)      ->", geo.analyze(line_ab), " grades", line_ab.grades)
print("join(line_ab, pC) ->", geo.analyze(plane_abc), " grades", plane_abc.grades)


join(pA, pB)      -> Line(org=Point(0.50, 0.50, -0.00), dir=Dir(0.71, -0.71, 0.00))  grades [2]
join(line_ab, pC) -> Plane(pt=Point(0.33, 0.33, 0.33), n=Dir(-0.58, -0.58, -0.58))  grades [1]


Note that the **outer product** `∧` gives the same intersection result as `meet()`
for planes — wedging two planes is their intersection line and wedging three is
their point.


In [9]:
# Outer product: the intersection for independent blades, zero for identical blades
print("plx ^ ply          ->", geo.analyze(plx ^ ply), " grades", (plx ^ ply).grades)
print("plx ^ ply ^ plz    ->", geo.analyze(plx ^ ply ^ plz), " grades", (plx ^ ply ^ plz).grades)
print("plx ^ plx is zero  :", (plx ^ plx).is_zero, "   (identical planes)")


plx ^ ply          -> Line(org=Point(0.00, 0.00, 0.00), dir=Dir(0.00, -0.00, 1.00))  grades [2]
plx ^ ply ^ plz    -> Point(-0.00, -0.00, 2.00)  grades [3]
plx ^ plx is zero  : True    (identical planes)


## 7. Euclidean motions — rotors, translators, motors

A **versor** is a product of invertible vectors; applied via the sandwich product
`V · x · ~V` it is a Euclidean motion. PGA3 unifies:

- **rotations** — the plane-based `Rotor` (scalar + Euclidean bivector),
- **translations** — `Translator` (scalar + `e0`-bivector),
- their composition, the **motor** — a rigid motion that rotates *and* translates
  in a single versor.


In [10]:
rot   = geo.create(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))
trans = geo.create(Translator(vector=Direction(2, 0, 0)))
motor = geo.create(Motor(
    rotor=Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)),
    translator=Translator(vector=Direction(2, 0, 0)),
))

rot.show("rotor (90° about z)")
print("grades", rot.grades)
print()
trans.show("translator (by 2 e1)")
print("grades", trans.grades)
print()
motor.show("motor (rotation followed by translation)")
print("grades", motor.grades)


rotor (90° about z): 0.7071 - 0.7071 e12

grades [0, 2]



translator (by 2 e1): 1 - e01

grades [0, 2]



motor (rotation followed by translation): 0.7071 - 0.7071 e01 + 0.7071 e02 - 0.7071 e12

grades [0, 2]


In [11]:
P = geo.create(Point(1, 0, 0))

print("P                  ->", geo.analyze(P))
print("rotor applied      ->", geo.analyze(PGA.vp(rot, P)))
print("translator applied ->", geo.analyze(PGA.vp(trans, P)))
print("motor applied      ->", geo.analyze(PGA.vp(motor, P)))
print()

# A motor analyzed by the geometry submodule reads back as a rotation about an
# offset origin (a GeneralRotor) — the translation part is folded into the pivot.
print("analyze(motor) ->", geo.analyze(motor))


P                  -> Point(1.00, -0.00, -0.00)
rotor applied      -> Point(-0.00, 1.00, -0.00)
translator applied -> Point(3.00, -0.00, -0.00)
motor applied      -> Point(2.00, 1.00, -0.00)

analyze(motor) -> GenRotor(90.0° about Dir(0.00, 0.00, 1.00) at Point(1.00, 1.00, 0.00))


## 8. Reflections — unified with motions

A reflection is also a versor. In PGA3 the reflection "blade" is simply the entity
blade itself, so points, lines, and planes all act as reflectors via the same
sandwich product — the same framework that produces rotations and translations.


In [12]:
q = geo.create(Point(1, 2, 3))

rp  = geo.create(ReflectionPlane(Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1))))
rl  = geo.create(ReflectionLine(Line(origin=Point(0, 0, 0), direction=Direction(0, 1, 0))))
rpt = geo.create(ReflectionPoint(Point(0, 0, 0)))

print("reflection in plane z=0    ->", geo.analyze(PGA.vp(rp, q)))
print("reflection in line y-axis ->", geo.analyze(PGA.vp(rl, q)))
print("reflection in origin       ->", geo.analyze(PGA.vp(rpt, q)))


reflection in plane z=0    -> Point(1.00, 2.00, -3.00)
reflection in line y-axis -> Point(-1.00, 2.00, -3.00)
reflection in origin       -> Point(-1.00, -2.00, -3.00)


The reflection "blades" are the same OPNS entity blades as before — a reflection
in the plane `z=0` is the blade `e3`, a reflection in the `y`-axis line is the
bivector `e31`, and a reflection in the origin is the trivector `e123`.


In [13]:
print("ReflectionPlane(z=0)  ->", rp,  " grades", rp.grades)
print("ReflectionLine(y-axis) ->", rl, " grades", rl.grades)
print("ReflectionPoint(origin)->", rpt, " grades", rpt.grades)


ReflectionPlane(z=0)  -> e3  grades [1]
ReflectionLine(y-axis) -> e31  grades [2]
ReflectionPoint(origin)-> e123  grades [3]


## 9. IPNS / OPNS and the Gunn/Dorst e0_recip

Like the other algebras, PGA3 reads the OPNS/IPNS interpretation from the mutable
`PGA.opns` flag (default `True`). Switching to IPNS flips the grades:

| Entity | OPNS (default) | IPNS |
|---|---|---|
| Point  | grade 3 | grade 1 (`x·e1 + y·e2 + z·e3 + e0`) |
| Line   | grade 2 | grade 2 (self-dual) |
| Plane  | grade 1 | grade 3 |


In [14]:
PGA.opns = False
p_ipns  = geo.create(Point(1, 2, 3))
pl_ipns = geo.create(Plane(point=Point(0, 0, 3), normal=Direction(0, 0, 1)))

p_ipns.show("IPNS point")
print("grades", p_ipns.grades)
print()
pl_ipns.show("IPNS plane")
print("grades", pl_ipns.grades)
print()

# Round-trips still agree
print("analyze(p_ipns)  ->", geo.analyze(p_ipns))
print("analyze(pl_ipns) ->", geo.analyze(pl_ipns))
PGA.opns = True


IPNS point: e0 + e1 + 2 e2 + 3 e3

grades [1]



IPNS plane: e021 - 3 e123

grades [3]

analyze(p_ipns)  -> Point(1.00, 2.00, 3.00)
analyze(pl_ipns) -> Plane(pt=Point(-0.00, -0.00, 3.00), n=Dir(0.00, 0.00, -1.00))


## 10. Visual examples

Two planes intersect in a line, three in a point. `pytanga.viz` analyses raw
multivectors on the way into the viewer, honouring `PGA.opns`. Viewer setup is
covered in [Part II — Visualization](../../visualization/).


In [15]:
from pytanga.viz import LineStyle, PlaneStyle, PointStyle, Visualizer

plx = PGA.e1                 # plane x = 0
ply = PGA.e2                 # plane y = 0
plz = PGA.e3 - 2 * PGA.e0    # plane z = 2

L = PGA.meet(plx, ply)                        # x=0 ∩ y=0 → the z-axis line
L.show("L")
P = PGA.meet(PGA.meet(plx, ply), plz)          # ∩ z=2    → the point (0, 0, 2)

viz = Visualizer(title="PGA3 — planes, their intersection line and point")
viz.add(plx, color="#4488ff", opacity=0.35, style=PlaneStyle(extent=5.0), label="x = 0")
viz.add(ply, color="#44cc44", opacity=0.35, style=PlaneStyle(extent=5.0), label="y = 0")
viz.add(plz, color="#cc6666", opacity=0.35, style=PlaneStyle(extent=5.0), label="z = 2")
viz.add(L, color="#ffcc00", style=LineStyle(length=5.0, thickness=0.05), label="intersection line")
viz.add(P, color="#B818B2", style=PointStyle(size=0.15), label="intersection point")

viz.display_snapshot()


L: e12

In [16]:
# A motor (rotation + translation) applied to a point.
# The motor is added without a label: its analyzed form is a GeneralRotor.
motor = geo.create(Motor(
    rotor=Rotor(angle=math.pi / 2, axis=Direction(1, 1, 1)),
    translator=Translator(vector=Direction(2, 0, 0)),
))
P = geo.create(Point(1, 0, 0))
P_moved = PGA.vp(motor, P)

viz2 = Visualizer(title="PGA3 — a motor (rotation + translation) applied to a point")
viz2.add(P, color="#ff4444", style=PointStyle(size=0.15), label="P")
viz2.add(P_moved, color="#44ff44", style=PointStyle(size=0.15), label="motor · P")
viz2.add(motor, color="#ff66cc")

viz2.display_snapshot()


Export both scenes as self-contained HTML:


In [17]:
import os

os.makedirs("_output/07_pga3", exist_ok=True)
viz.export_snapshot("_output/07_pga3/intersections.html", overwrite=True)
viz2.export_snapshot("_output/07_pga3/motor.html", overwrite=True)
print("Exported _output/07_pga3/intersections.html")
print("Exported _output/07_pga3/motor.html")


Exported _output/07_pga3/intersections.html
Exported _output/07_pga3/motor.html


## 11. Summary & next steps

You now know the plane-based model `BasisPGA3`:

| Concept | API |
|---|---|
| Null embedding | `PGA.e0` (`e0² = 0`), `PGA.e0_recip` |
| Plane / line / point | `geo(Plane(...))` / `geo(Line(...))` / `geo(Point(...))` |
| Plane as vector | `n + d·e0` (e.g. `e3 - 3·e0`) |
| Line as bivector | `plane1 ^ plane2` |
| Point as trivector | `plane1 ^ plane2 ^ plane3` |
| Intersection | `PGA.meet(A, B)` |
| Union of objects | `PGA.join(A, B)` |
| Incidence | `point ^ plane` → zero |
| Motions | `geo(Rotor/Translator/Motor(...))` → `PGA.vp(V, x)` |
| Reflections | `geo(ReflectionPlane/ReflectionLine/ReflectionPoint(...))` |

**Where to go next:**

- [**08 · Duality & Complements**](../08_duality/) — the `dual()`/`ldual()`/
  `complement()` family behind `join()` and `meet()`.
- [**14 · Geometry Submodule**](../14_geometry/) — the full entity/operator data
  model and round-trip pipeline used throughout this tutorial.
